# Project Title : Drone Detection System for Surveillance

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Problem Statement : 

### With the increasing use of drones in unauthorized and potentially harmful activities, detecting drones in surveillance environments has become a critical challenge. Traditional monitoring systems struggle to differentiate drones from similar entities such as birds or aircraft, especially under noisy environmental conditions.

### This project aims to develop an intelligent multi-modal detection system that identifies drones using both:

### Visual data (video frames)

### Acoustic signals (audio recordings)

### The system will:

* ### Detect and classify flying objects (drone vs bird vs aircraft)

* ### Use audio signals to validate or reject visual detections

* ### Reduce false positives caused by birds or environmental noise

* ### Generate prioritized alerts for potential threats

In [2]:
import os

audio_path = "/kaggle/input/datasets/partar/drone-detect-dont-upload/Audio"
video_path = "/kaggle/input/datasets/partar/drone-detect-dont-upload/Video"

audio_files = os.listdir(audio_path)
video_files = os.listdir(video_path)

audio_data = []
video_data = []

# Extract Audio files.

for file in audio_files:
    if file.endswith((".wav", '.mp3')):
        label = file.split('_')[0]
        full_path = os.path.join(audio_path, file)
        audio_data.append((full_path, label))

# Extract Video files.

for file in video_files:
    if file.endswith('.mp4'):
        label = file.split('_')[1]
        full_path = os.path.join(video_path, file)
        video_data.append((full_path, label))

#### **Extract both audio and video file path and save it to "audio_data" and "video_data" respectivly.**

#### **This files are saved in the form of list.**

In [3]:
# Created Datasets.

audio_df = pd.DataFrame(audio_data, columns = ['Path', 'Class'])
video_df = pd.DataFrame(video_data, columns = ['Path', 'Class'])

In [4]:
# make Label "Threat" if class is "DRONE" else "Non-Threat" in both audio and video data

audio_df['Label'] = audio_df['Class'].apply(lambda x : "Threat" if x == 'DRONE' else "Non-Threat")
video_df['Label'] = video_df['Class'].apply(lambda x : "Threat" if x == 'DRONE' else "Non-Threat")

#### **Make label "Threat" if drone detected otherwise "Non-Threat"**

In [5]:
audio_df.head()

,Path,Class,Label
0,/kaggle/input/datasets/partar/drone-detect-don...,BACKGROUND,Non-Threat
1,/kaggle/input/datasets/partar/drone-detect-don...,DRONE,Threat
2,/kaggle/input/datasets/partar/drone-detect-don...,AIRPLANE,Non-Threat
3,/kaggle/input/datasets/partar/drone-detect-don...,DRONE,Threat
4,/kaggle/input/datasets/partar/drone-detect-don...,HELICOPTER,Non-Threat


In [6]:
audio_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Path    112 non-null    object
 1   Class   112 non-null    object
 2   Label   112 non-null    object
dtypes: object(3)
memory usage: 2.8+ KB


#### There are just 112 audio data in audio file.

In [7]:
video_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 261 entries, 0 to 260
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Path    261 non-null    object
 1   Class   261 non-null    object
 2   Label   261 non-null    object
dtypes: object(3)
memory usage: 6.2+ KB


#### There are just 261 video data in video file.

## **Data Augmentation for Audio.**

**We have realy less amount of data so we need to augment data from existing data.**

In [8]:
import librosa

def add_noise(signal):
    noise = np.random.randn(len(signal))
    return signal + 0.005 * noise

def shift_audio(signal):
    shift = np.random.randint(len(signal) // 10)
    return np.roll(signal, shift)

def pitch_shift(signal, sr):
    return librosa.effects.pitch_shift(signal, sr = sr, n_steps = 2)

def time_stretch(signal):
    return librosa.effects.time_stretch(signal, rate = 1.2)

def change_volume(signal):
    return signal * np.random.uniform(0.8, 1.5)

**add_noise :** Generate a random noise and added to the original audio.

**shift_audio :** Moves audio forward or backward in time.

**pitch_shift :** Change frequency pitch (low pitch -> high pitch)

**time_stretch :** Change speed of audio.

**chnage_volume :** Increse / decrease amplitude.

In [9]:
import random

def augment_audio(signal, sr):
    augmentations = [
        lambda x: add_noise(x),
        lambda x: shift_audio(x),
        lambda x: pitch_shift(x, sr),
        lambda x: time_stretch(x),
        lambda x: change_volume(x)
    ]

    # randomly choose 2–3 augmentations
    num_aug = random.randint(2, 3)
    selected = random.sample(augmentations, num_aug)

    for aug in selected:
        signal = aug(signal)

    return signal

In [10]:
def fix_length(signal, target_len):
    if len(signal) > target_len:
        return signal[:target_len]
    else:
        return np.pad(signal, (0, target_len - len(signal)))

In [11]:
# Audio Processing.

def extract_feature(file_path, augment = True):
    signal, sr = librosa.load(file_path, sr=22100)
    original_len = len(signal)

    if augment:
        signal = augment_audio(signal, sr)
        signal = fix_length(signal, original_len)

    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
    mfcc = np.mean(mfcc.T, axis=0)

    return mfcc

In [12]:
X, y = [], []

for i in range(len(audio_df)):
    file_path = audio_df.iloc[i]['Path']
    label = audio_df.iloc[i]['Label']
    X.append(extract_feature(file_path, augment=False))
    y.append(label)

    # create 2 augmented samples
    for _ in range(2):  
        X.append(extract_feature(file_path, augment=True))
        y.append(label)

X = np.array(X)
y = np.array(y)

In [13]:
len(X)

336

#### **Above code will converts the audio amplitude into numerical data. And also create new audio from existing one using some augmentation techniques.**

In [14]:
# Encode label data.

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [15]:
# Split data with 80-20 ratio.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 101)

In [16]:
from sklearn.ensemble import GradientBoostingClassifier

audio_model = GradientBoostingClassifier()
audio_model.fit(X_train, y_train)
y_pred = audio_model.predict(X_test)
print('Confusion Matrix : \n', classification_report(y_test, y_pred))

Confusion Matrix : 
               precision    recall  f1-score   support

           0       0.94      0.92      0.93        52
           1       0.76      0.81      0.79        16

    accuracy                           0.90        68
   macro avg       0.85      0.87      0.86        68
weighted avg       0.90      0.90      0.90        68



#### **I have checked for different algorithms like Random forest, Gradient boost, adaboost, SVM, But Gradient Boost returns good accuracy among them**

## Data Augmentation for Video.

In [17]:
import cv2
import numpy as np
import random

def horizontal_flip(frames):
    return [cv2.flip(f, 1) for f in frames]

def brightness_contrast(frames):
    alpha = random.uniform(0.8, 1.2)  # contrast
    beta = random.randint(-20, 20)    # brightness
    return [cv2.convertScaleAbs(f, alpha=alpha, beta=beta) for f in frames]

def gaussian_blur(frames):
    return [cv2.GaussianBlur(f, (5, 5), 0) for f in frames]

**horizontal_flip :** Flip each frame left to right or right to left.

**brightness_contrast :** Increase brightness or constrast of each frame.

**gaussian_blur :** This smooths images like remove sharp edges.

In [18]:
def augment_video(frames):
    augmentations = [
        horizontal_flip,
        brightness_contrast,
        gaussian_blur
    ]

    num_aug = 1
    selected = random.sample(augmentations, num_aug)

    for aug in selected:
        frames = aug(frames)

    return frames

In [19]:
def extract_frame(video_path, max_frames=50, augment=False):
    cap = cv2.VideoCapture(video_path)
    frames = []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total_frames // max_frames)

    count = 0
    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        if count % step == 0:
            frame = cv2.resize(frame, (64, 64))
            frame = frame.astype('float32') / 255.0
            frames.append(frame)

        count += 1

    cap.release()

    # padding
    if len(frames) < max_frames:
        padding = [np.zeros((64, 64, 3))] * (max_frames - len(frames))
        frames.extend(padding)

    frames = frames[:max_frames]

    # APPLY AUGMENTATION
    if augment:
        frames = augment_video(frames)

    # CRITICAL: CLIP VALUES AFTER AUGMENTATION
    frames = [np.clip(f, 0.0, 1.0) for f in frames]

    return np.array(frames)

In [20]:
X, y = [], []

for i in range(len(video_df)):
    video_path = video_df.iloc[i]['Path']
    label = video_df.iloc[i]['Label']

    # Original sample
    frames = extract_frame(video_path, augment=False)
    X.append(frames)
    y.append(label)

    # create 2 augmented versions
    frames_aug = extract_frame(video_path, augment=True)
    X.append(frames_aug)
    y.append(label)

X = np.array(X)
y = np.array(y)

#### **Above code will extract each video from video path, then make video into normalized image frame**

In [21]:
# Encode label data to convert into numerical form.

encode = LabelEncoder()
y = encode.fit_transform(y)

In [22]:
# Split data with 80-20 ratio.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 101)

In [23]:
# Shape of our data

X_train.shape

(417, 50, 64, 64, 3)

#### 417 : no. of data in training dataset (X_train)

#### 50 : no. frame per data (video)

#### 64, 64 : size of frame or image (64 * 64)

#### 3 : colored image (RGB)

In [24]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

class_weights = dict(enumerate(class_weights))

create class_weights to tell our model about imbalaceness of distrubtion of class in our data.

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TimeDistributed, Dense, GlobalAveragePooling2D, SimpleRNN, Bidirectional, Conv2D, BatchNormalization, MaxPooling2D, Dropout
from tensorflow.keras.optimizers import Adam

2026-03-26 17:37:06.051336: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774546626.225980      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774546626.272133      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774546626.698531      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774546626.698578      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774546626.698581      24 computation_placer.cc:177] computation placer alr

#### We need two different models to train model on video data. 

#### CNN + RNN : CNN to deal with spatial data and RNN to deal with sequential data. Because video is just sequence of images for machine.

#### Video frames will first processed by CNN model and then precess in RNN.

In [26]:
# Model training.

cnn_model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape = (64, 64, 3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    
    GlobalAveragePooling2D()
])
    
video_model = Sequential([
    TimeDistributed(cnn_model, input_shape = (20, 64, 64, 3)),
    Bidirectional(SimpleRNN(128, activation = 'leaky_relu', return_sequences = True)),
    Bidirectional(SimpleRNN(64, activation = 'leaky_relu')),
    Dense(128, activation = 'tanh'),
    Dense(2, activation = 'softmax')
])

video_model.compile(optimizer = Adam(learning_rate = 4e-5, weight_decay = 1e-4),
                   loss = 'sparse_categorical_crossentropy',
                   metrics = ['accuracy'])

video_model.fit(X_train, y_train, epochs = 5, validation_split = 0.3, class_weight=class_weights)
y_pred = video_model.predict(X_test)
y_pred = np.argmax(y_pred, axis = 1)

print('classification report : \n', classification_report(y_test, y_pred))

I0000 00:00:1774546646.798937      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/5


I0000 00:00:1774546688.047536    2192 service.cc:152] XLA service 0x66a90730 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774546688.047579    2192 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1774546697.849368    2192 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1774546731.680449    2192 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10/10 ━━━━━━━━━━━━━━━━━━━━ 148s 7s/step - accuracy: 0.6899 - loss: 0.5745 - val_accuracy: 0.7381 - val_loss: 0.5703
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 208ms/step - accuracy: 0.8106 - loss: 0.3810 - val_accuracy: 0.9048 - val_loss: 0.3276
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 206ms/step - accuracy: 0.8451 - loss: 0.3115 - val_accuracy: 0.9206 - val_loss: 0.3276
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 207ms/step - accuracy: 0.9119 - loss: 0.2857 - val_accuracy: 0.9048 - val_loss: 0.3153
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 206ms/step - accuracy: 0.8890 - loss: 0.2690 - val_accuracy: 0.9206 - val_loss: 0.3020
4/4 ━━━━━━━━━━━━━━━━━━━━ 22s 4s/step
classification report : 
               precision    recall  f1-score   support

           0       0.88      0.93      0.90        75
           1       0.80      0.67      0.73        30

    accuracy                           0.86       105
   macro avg       0.84      0.80      0.82       105
weighted avg       0.85      0.86      0.

I have check some hyperparameters but i found below with higher accuracy:

1. **model :** SimpleRNN
2. **learning rate :** 410^-5
3. **optimizer :** Adam
4. **weight decay :** 10^-4
5. **activation funcition :** relu (for CNN), leaky_relu (for RNN), tanh (for Dense)

In [27]:
# save both the models.

# import joblib

# video_model.save('video_model.keras')
# joblib.dump(audio_model, 'audio_model.pkl')

#### Save both models to use it further.